In [1]:
import geopandas as gpd
import pandas as pd
import folium
from pathlib import Path

BASE = Path(r"C:\Users\msi\OneDrive\Desktop\corridor-intelligence-platform")

In [2]:
corridors = gpd.read_file(BASE / "data/processed/corridors/combinado_final.geojson")
buf10     = gpd.read_file(BASE / "data/processed/buffers/Buffer__10km.geojson")
events    = gpd.read_file(BASE / "data/synthetic/synthetic_corridor_events.geojson")

risk = pd.DataFrame({
    "corridor_id":   [1, 2, 3],
    "corridor_name": ["Neuquén-Añelo", "Añelo-Rincón", "Neuquén-Catriel"],
    "eventos":       [40, 25, 15],
    "score":         [100.0, 61.3, 36.6],
    "risk_level":    ["ALTO", "ALTO", "MEDIO"]
})

print(risk)

   corridor_id    corridor_name  eventos  score risk_level
0            1    Neuquén-Añelo       40  100.0       ALTO
1            2     Añelo-Rincón       25   61.3       ALTO
2            3  Neuquén-Catriel       15   36.6      MEDIO


In [3]:
color_map = {"ALTO": "#e74c3c", "MEDIO": "#f39c12", "BAJO": "#27ae60"}

mapa = folium.Map(location=[-38.5, -68.5], zoom_start=7)

# Buffer 10km coloreado por riesgo
for _, corridor in corridors.iterrows():
    cid  = corridor["Id"]
    info = risk[risk["corridor_id"] == cid].iloc[0]
    color = color_map[info["risk_level"]]

    # Buffer del corredor
    buf = buf10[buf10["Id"] == cid]
    folium.GeoJson(
        buf,
        style_function=lambda x, c=color: {
            "fillColor": c, "color": c, "fillOpacity": 0.3, "weight": 1
        }
    ).add_to(mapa)

    # Línea del corredor
    folium.GeoJson(
        corridor["geometry"].__geo_interface__,
        style_function=lambda x, c=color: {"color": c, "weight": 4}
    ).add_to(mapa)

# Eventos sintéticos
type_colors = {
    "protesta":   "#8e44ad",
    "accidente":  "#e74c3c",
    "clima":      "#2980b9",
    "congestion": "#f39c12",
    "obra_vial":  "#7f8c8d"
}

for _, ev in events.iterrows():
    folium.CircleMarker(
        location=[ev["latitude"], ev["longitude"]],
        radius=5,
        color=type_colors.get(ev["event_type"], "#333"),
        fill=True,
        fill_opacity=0.8,
        popup=f"{ev['event_type']} | {ev['subtipo']} | Severidad: {ev['severidad']}"
    ).add_to(mapa)

folium.LayerControl().add_to(mapa)
mapa

In [4]:
print("── Ranking de Riesgo Operacional ────────────────────")
print(f"{'Corredor':<25} {'Eventos':>8} {'Score':>7} {'Riesgo':>8}")
print("─" * 52)
for _, row in risk.sort_values("score", ascending=False).iterrows():
    print(f"{row['corridor_name']:<25} {row['eventos']:>8} {row['score']:>7} {row['risk_level']:>8}")

── Ranking de Riesgo Operacional ────────────────────
Corredor                   Eventos   Score   Riesgo
────────────────────────────────────────────────────
Neuquén-Añelo                   40   100.0     ALTO
Añelo-Rincón                    25    61.3     ALTO
Neuquén-Catriel                 15    36.6    MEDIO


In [5]:
out = BASE / "docs/maps/03_risk_engine.html"
out.parent.mkdir(parents=True, exist_ok=True)
mapa.save(str(out))
print(f"✓ Mapa guardado: {out}")

✓ Mapa guardado: C:\Users\msi\OneDrive\Desktop\corridor-intelligence-platform\docs\maps\03_risk_engine.html
